# CubeSat Geometry

Builds the DarkNESS 6U double-deployable geometry, realizes it into the body frame, renders the 3D scene, and saves `outputs/spacecraft.json` + `outputs/body_roles.json` for downstream notebooks.

Re-run this notebook any time the geometry, mechanism state, or mount rotation changes.

In [1]:
import json
import math
from pathlib import Path
import numpy as np

from geometry import build_6u_double_deployable, mount
from geometry.CubeSat import scene
from geometry.CubeSat.inspect import print_surface_summary


In [2]:
# -- Geometry parameters ---------------------------------------------------
leaf_y = 0.2263   # panel-leaf dimension along body Y [m]
leaf_z = 0.3405   # panel-leaf dimension along body Z [m]
bus_patch_shape  = (24, 16)
wing_patch_shape = (24, 16)

mechanism_state = {
    'wing_port_inner_angle':       math.pi / 2.0,
    'wing_port_outer_angle':        math.pi,
    'wing_starboard_inner_angle':  -math.pi / 2.0,
    'wing_starboard_outer_angle':  -math.pi,
}
mount_rotation = mount('+Y', '+Z', '+Z', '+X')
mount_offset   = np.zeros(3)
outputs_dir    = Path('outputs')

cubesat = build_6u_double_deployable(
    leaf_y=leaf_y, leaf_z=leaf_z,
    bus_patch_shape=bus_patch_shape,
    wing_patch_shape=wing_patch_shape,
)

In [3]:
# -- Realize into body frame -----------------------------------------------
realized = cubesat.realize(
    mechanism_state,
    mount_rotation=mount_rotation,
    mount_offset=mount_offset,
)
print_surface_summary(realized)

builder id               body role              center [m]                     normal   size [m]           patches tags
-----------------------------------------------------------------------------------------------------------------------
bus_+X                   body +Y bus face       (0.0, 0.05, 0.0)               +Y       0.3405 x 0.2263    24 x 16 bus, +X
bus_-X                   body -Y bus face       (0.0, -0.05, 0.0)              -Y       0.3405 x 0.2263    24 x 16 bus, -X
bus_+Y                   body +Z bus face       (0.0, 0.0, 0.1132)             +Z       0.3405 x 0.1000    24 x 16 bus, +Y
bus_-Y                   body -Z bus face       (0.0, 0.0, -0.1132)            -Z       0.3405 x 0.1000    24 x 16 bus, -Y
bus_+Z                   body +X bus face       (0.1703, 0.0, 0.0)             +X       0.1000 x 0.2263    24 x 16 bus, +Z
bus_-Z                   body -X bus face       (-0.1703, 0.0, 0.0)            -X       0.1000 x 0.2263    24 x 16 bus, -Z
wing_port_inner       

In [4]:
# -- 3D scene --------------------------------------------------------------
scene(realized, title='DarkNESS 6U — realized geometry').show()

In [5]:
# -- Save outputs ----------------------------------------------------------
outputs_dir.mkdir(exist_ok=True)

saved = realized.to_json(outputs_dir / 'spacecraft.json')
print(f'Saved {saved}')

# Body-role map: body-axis label -> surface name
# Derived from realized normals; regenerates automatically if mount changes.
def _body_label(normal):
    idx = int(np.abs(normal).argmax())
    return ('+' if normal[idx] > 0 else '-') + 'XYZ'[idx]

body_roles = {_body_label(s.normal): s.name for s in realized.by_tag('bus')}
body_roles_path = outputs_dir / 'body_roles.json'
body_roles_path.write_text(json.dumps(body_roles, indent=2), encoding='utf-8')
print(f'Saved {body_roles_path}')

print()
print(f'  {"body axis":<10}  {"surface name":<16}  size [m]')
print(f'  {"-"*10}  {"-"*16}  --------')
for label in sorted(body_roles):
    s = realized.by_name(body_roles[label])
    print(f'  {label:<10}  {s.name:<16}  {s.width:.4f} x {s.height:.4f}')

Saved outputs\spacecraft.json
Saved outputs\body_roles.json

  body axis   surface name      size [m]
  ----------  ----------------  --------
  +X          bus_+Z            0.1000 x 0.2263
  +Y          bus_+X            0.3405 x 0.2263
  +Z          bus_+Y            0.3405 x 0.1000
  -X          bus_-Z            0.1000 x 0.2263
  -Y          bus_-X            0.3405 x 0.2263
  -Z          bus_-Y            0.3405 x 0.1000
